# Function Callingfor MCP

**Module 10 · Lesson 10.1**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Tool-Use Loop
- Three Providers, One Concept
- Schema Design — the Reliability Lever
- tool_choice & Parallel Calls
- A Universal ToolExecutor
- Don’t Trust the Args — or the Results

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic openai -q

# Reproducibility - the lesson uses seeded random values throughout

## The Model That Couldn’t Check the Weather

> 💡 **Info**
>
> Ask a bare language model “what’s the weather in Hyderabad right now?’’ and it will answer confidently — something like “around 30°C and sunny.’’ It has no idea. It has no thermometer, no internet, no clock; it just predicts plausible text. Now give it ONE thing: a description of a `get_weather(city)` tool. Ask again, and instead of guessing it stops and emits a structured request: `get_weather("Hyderabad")`. Your code runs the real API, hands back `34°C`, and the model reports the TRUE number. That single shift — from guessing text to requesting an action — is function calling, and it is what turns a chatbot into something that can actually DO things.

### 🎯 What you will be able to do after this lesson

- **Build** the provider-agnostic tool-use loop and run it on the current Anthropic, OpenAI, and google-genai APIs (with the right field names: tool_use/tool_result, tool_calls, automatic function calling).

- **Compare** the three provider shapes, the tool_choice modes (auto/required/forced/none), and parallel vs sequential calls — and design schemas (descriptions, enums, required) that make tool-calling reliable.

- **Operate** the safety layer: strict mode + validation, return-error-to-model self-correction (never silent-fallback), and least-privilege access against prompt-injection-via-tool-results.

- **Evaluate** why function calling leads to **MCP** (N×M → N+M via JSON-RPC 2.0) and the India path (Sarvam + Bhashini + translate-first).

> 📦 **Info**
>
> ✅ Before you startThis opens **Module 10**. Module 7 introduced the LLM APIs; here you master the tool-use primitive that the rest of the module — **MCP**, which we get to in Lesson 10.2 onward, and tool orchestration — is built on. You should be comfortable with the chat-completions request shape and JSON.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🍽️ **Analogy**
>
> **Function calling is a waiter taking your order.** You tell the waiter (the model) what you want in plain language; the waiter writes a structured ticket (the tool call) and hands it to the kitchen (YOUR code), which actually cooks. The waiter never cooks and never invents the dish — they carry the real result back to you. **Where it breaks down:** a good waiter doesn’t serve a garbled ticket raw — a malformed order goes back to be rewritten (validation + self-correction), not sent to the kitchen as-is.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“The model runs the tool.”** It doesn’t. It emits a request (name + args); YOUR code executes and returns the result. The model is a planner, not a runtime.
> - **“MCP replaces function calling.”** MCP *standardizes* it (JSON-RPC 2.0). Under the hood, an MCP tool call is still a function call.

> 💡 **Info**
>
> ⚠️ Anti-patternWhen the model returns malformed arguments, **silently defaulting them to an empty dict and running the tool anyway**. The tool executes with wrong inputs and returns a confidently wrong result that no one flagged — the single most common production tool-calling failure.

---

## 🎯 Concept 1: The Tool-Use Loop

### The Tool-Use Loop

Four steps: you send tools, the model REQUESTS a call, your code runs it, you send the result back.

Strip away the provider differences and every tool-calling system is the same four-step loop. (1) You send the user’s message plus your tool schemas. (2) Instead of answering, the model returns a structured *request*: a tool name and JSON arguments. (3) **Your** code runs the real function and captures the result. (4) You send the result back and the model writes the final natural-language answer. The model never executes anything — it only decides *what* to call.

> 📡 **Analogy**
>
> It’s a **drive-through intercom**. You speak your order; the attendant (model) doesn’t cook — they punch it into the till (the tool call), the kitchen makes it, and it comes back to the window. The intercom is a router of intent, not a chef.

You ask a tool-enabled model “what’s 14999 plus about 18% GST?”. What does the model return on the FIRST step?

**📝 `01_loop.py`** - *runnable*

In [ ]:
# THE TOOL-USE LOOP - the model REQUESTS a call; YOUR code runs it and feeds the result back.
import ast, operator
def eval_safe(expr):                              # tiny safe arithmetic (no eval, no builtins)
    OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):    return OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval").body)

def mock_model_pick(msg):                          # stands in for the LLM (keyword router)
    m = msg.lower()
    if "weather" in m: return {"tool": "get_weather", "args": {"city": "Hyderabad"}}
    if any(c.isdigit() for c in m): return {"tool": "calculate", "args": {"expr": "14999*1.18"}}
    return {"tool": None, "answer": "I can answer that directly."}

def run_tool(name, args):
    if name == "get_weather": return {"temp_c": 34, "cond": "Sunny"}
    if name == "calculate":   return {"result": round(eval_safe(args["expr"]), 2)}

def tool_use_loop(msg):
    print(f"  1. user -> model:   {msg!r}")
    step = mock_model_pick(msg)
    if not step["tool"]:
        print(f"  2. model -> answer: {step['answer']} (no tool needed)"); return
    print(f"  2. model REQUESTS:  {step['tool']}({step['args']})   <- a request, not an execution")
    print(f"  3. YOUR CODE runs it -> {run_tool(step['tool'], step['args'])}")
    print(f"  4. result -> model -> final natural-language answer")

tool_use_loop("what's the weather in Hyderabad?")
tool_use_loop("what is 14999 plus 18% GST?")

# Output:
#   1. user -> model:   "what's the weather in Hyderabad?"
#   2. model REQUESTS:  get_weather({'city': 'Hyderabad'})   <- a request, not an execution
#   3. YOUR CODE runs it -> {'temp_c': 34, 'cond': 'Sunny'}
#   4. result -> model -> final natural-language answer
#   1. user -> model:   'what is 14999 plus 18% GST?'
#   2. model REQUESTS:  calculate({'expr': '14999*1.18'})   <- a request, not an execution
#   3. YOUR CODE runs it -> {'result': 17698.82}
#   4. result -> model -> final natural-language answer

- `mock_model_pick` stands in for the LLM — the point is the LOOP shape, not the routing.
- Step 2 prints a REQUEST (name + args); the model has NOT run anything yet.
- Step 3 is `run_tool` — YOUR code, executing the real function and returning a result.
- The safe `eval_safe` (ast-based, no `eval`) previews Step 5 — never run model-provided expressions with `eval`.

#### 💡 What just happened

⚡ What just happened? Every provider implements this exact loop — only the field names differ. The model is a **planner** that emits `get_weather(city="Hyderabad")`; your code is the **executor** that runs it and returns `34°C`. Keep that split clear and the rest of tool use is bookkeeping. And note this loop **repeats**: you append the model’s tool-request turn AND the tool_result to `messages`, then call the model again — it may request another tool, and you keep going until it stops requesting and writes the answer. One turn is the unit; real tasks chain several.

- Click through the 5 message hops (the same 4 steps above; the last one, send-result then model-replies, is shown as two hops)
- See what happens and the message shape at each step

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Three Providers, One Concept

### Three Providers, One Concept

Anthropic, OpenAI, and google-genai express the same tool three ways. Learn the field names once.

The concept is identical; the plumbing differs. **Anthropic** puts the schema under `input_schema` and returns `content` blocks with `type=="tool_use"`. **OpenAI** wraps it as `{type:"function", function:{parameters:...}}` and returns `message.tool_calls[]` (arguments come back as a JSON *string* you must parse). **google-genai** is simplest: you pass the Python function itself and automatic function calling runs the loop for you. Two migrations matter in 2026 — see the code.

> 🔌 **Analogy**
>
> It’s **three power plugs for the same voltage**. UK, EU, and US sockets look different, but the electricity is the same — you just need the right adapter. A universal executor (Step 5) is that travel adapter; MCP (Step 7) is the international standard that makes the adapters unnecessary.

Your 2024 Gemini code calls `GenerativeModel(tools=).start_chat(enable_automatic_function_calling=True)` on the google-genai SDK. What happens?

**📝 `02_schemas.py`** - *runnable*

In [ ]:
# THE SAME TOOL, THREE PROVIDER SHAPES (the concept is identical; the field names differ).
def anthropic_tool(name, desc, schema):
    return {"name": name, "description": desc, "input_schema": schema}          # -> content[].type=="tool_use"
def openai_tool(name, desc, schema):
    return {"type": "function", "function": {"name": name, "description": desc, "parameters": schema}}  # -> message.tool_calls[]
# google-genai: you pass the PYTHON FUNCTION itself; the SDK builds the schema from hints+docstring.

schema = {"type": "object",
          "properties": {"city": {"type": "string", "description": "City name"},
                         "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}},
          "required": ["city"]}
a = anthropic_tool("get_weather", "Get current weather for a city", schema)
o = openai_tool("get_weather", "Get current weather for a city", schema)
print("  Anthropic: schema key =", list(a.keys())[-1], "| response = content[].type == 'tool_use'")
print("  OpenAI:    schema key =", list(o['function'].keys())[-1], "| response = message.tool_calls[] (.arguments is a JSON string)")
print("  Google:    pass the python function; automatic function calling is ON by default")

# Output:
#   Anthropic: schema key = input_schema | response = content[].type == 'tool_use'
#   OpenAI:    schema key = parameters | response = message.tool_calls[] (.arguments is a JSON string)
#   Google:    pass the python function; automatic function calling is ON by default

- Anthropic keeps the JSON schema under `input_schema`; OpenAI under `parameters`; google-genai builds it from the function itself.
- The response shapes differ too: `content[].type=='tool_use'` vs `message.tool_calls[]` vs auto-handled.
- OpenAI’s `tool_calls[].function.arguments` is a JSON STRING — `json.loads` it before use.
- This field-name mismatch across N clients and M tools is exactly what MCP standardizes (Step 7).

**📝 `02b_real_loops.py current`** - *APIs*

In [ ]:
# THE REAL LOOPS (illustrative; need API keys). CURRENT 2026 APIs - note the migrations.
# Minimal example only - production code wraps each call in try/except with a timeout= (see 05_executor.py).
# ANTHROPIC
import anthropic
client = anthropic.Anthropic()                                   # ANTHROPIC_API_KEY
r = client.messages.create(model="claude-sonnet-4-6", max_tokens=500, tools=[anthropic_tool],
                           messages=[{"role":"user","content":"weather in Hyderabad?"}])
for block in r.content:
    if block.type == "tool_use":                                 # block.name, block.input, block.id
        result = run_tool(block.name, block.input)               # you reply with a tool_result block

# OPENAI (tools/tool_calls; legacy function_call is deprecated; strict=True for guaranteed JSON)
from openai import OpenAI
import json
oai = OpenAI()                                                   # OPENAI_API_KEY
resp = oai.chat.completions.create(model="gpt-5.5", tools=[openai_tool],
                                   messages=[{"role":"user","content":"weather in Hyderabad?"}])
for tc in (resp.choices[0].message.tool_calls or []):
    args = json.loads(tc.function.arguments)                     # arguments is a JSON STRING
    # append {"role":"tool","tool_call_id":tc.id,"content":...} and re-call

# GOOGLE (google-genai; NOT google.generativeai; automatic function calling is ON by default)
from google import genai
from google.genai import types
g = genai.Client()                                              # GOOGLE_API_KEY
chat = g.chats.create(model="gemini-2.5-flash",
                      config=types.GenerateContentConfig(tools=[get_weather]))  # pass the python func
print(chat.send_message("weather in Hyderabad?").text)          # SDK runs the tool for you

# Output: (illustrative - runs with the three API keys set)

#### 💡 What just happened

⚡ What just happened? Three shapes, one loop. The 2026 migrations: **google-genai** replaced `google.generativeai` (automatic function calling is on by default, no flag); **OpenAI** uses `tools`/`tool_calls` (the old `function_call` is deprecated) with `strict:true` for guaranteed-valid JSON, and the **Responses API** is the unified successor to Chat Completions (a newer OpenAI endpoint merging Chat Completions and Assistants; Chat Completions still works, so this code stays valid). Models move too — `claude-sonnet-4-6`, a current GPT-5.x, `gemini-2.5-flash`. The **tradeoff** of coding straight to a provider SDK is lock-in: switching vendors means rewriting the field names. The **alternative** is a universal executor (Step 5) that hides the shapes, and ultimately MCP (Step 7) which removes them entirely. Two details that bite in production: each `tool_result` must carry the *id* of the request it answers (`tool_use_id` / `tool_call_id`), and when you stream, tool-call arguments arrive as incremental deltas — buffer them until the block completes before `json.loads`, never parse partial fragments.

- Pick Anthropic / OpenAI / google-genai
- See the schema key and response field names

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Schema Design — the Reliability Lever

### Schema Design — the Reliability Lever

The model reads your descriptions to decide WHEN to call and your schema to fill the args. Vague schemas = unreliable calls.

Tool-calling accuracy is mostly a *schema* problem, not a model problem. The model uses each tool’s **description** to decide whether it’s relevant, and the **parameter schema** to fill in arguments. Four levers move reliability a lot: a specific description, `enum` for known values (so the model can’t invent one), truly-required fields marked `required`, and clear `verb_noun` names.

> 📝 **Analogy**
>
> A tool schema is a **job description**. “Weather tool’’ is a vague posting that attracts the wrong applicants; “Get the current weather for a specific city, unit in celsius or fahrenheit’’ tells the model exactly when to apply and what to bring. The clearer the posting, the better the hire — the tool gets called at the right time with the right args.

Two schemas differ only in that one has a rich description + an enum on `unit` and the other is `{{"description":"weather tool"}}`. Which calls more reliably?

**📝 `03_schema_score.py`** - *runnable*

In [ ]:
# SCHEMA QUALITY -> TOOL-CALL RELIABILITY. The model reads descriptions to decide WHEN to call.
def schema_score(tool):
    pts, notes = 0, []
    d = tool.get("description", "")
    if len(d) >= 20 and d.lower() not in ("weather tool", "tool"): pts += 30; notes.append("good description (+30)")
    else: notes.append("weak description (0) - the model reads this to decide WHEN to call")
    props = tool.get("input_schema", {}).get("properties", {})
    if any("enum" in p for p in props.values()): pts += 25; notes.append("enum constrains values (+25)")
    else: notes.append("no enum -> model may hallucinate values")
    if tool.get("input_schema", {}).get("required"): pts += 25; notes.append("required fields marked (+25)")
    else: notes.append("nothing required -> model may skip critical args")
    if "_" in tool["name"] and tool["name"].split("_")[0] in ("get","create","search","update","delete","list","send"):
        pts += 20; notes.append("verb_noun name (+20)")
    else: notes.append("unclear name")
    return pts, notes

good = {"name":"get_weather","description":"Get the current weather for a specific city",
        "input_schema":{"properties":{"unit":{"enum":["celsius","fahrenheit"]}},"required":["city"]}}
bad  = {"name":"weather","description":"weather tool","input_schema":{"properties":{"c":{"type":"string"}}}}
for label, t in [("GOOD schema", good), ("BAD schema", bad)]:
    s, notes = schema_score(t)
    print(f"  {label}: reliability score {s}/100")
    for n in notes: print(f"      - {n}")

# Output:
#   GOOD schema: reliability score 100/100
#       - good description (+30)
#       - enum constrains values (+25)
#       - required fields marked (+25)
#       - verb_noun name (+20)
#   BAD schema: reliability score 0/100
#       - weak description (0) - the model reads this to decide WHEN to call
#       - no enum -> model may hallucinate values
#       - nothing required -> model may skip critical args
#       - unclear name

- The GOOD schema scores 100: specific description (+30), enum (+25), required (+25), verb_noun name (+20).
- The BAD schema scores 0: “weather tool’’ tells the model nothing about WHEN to call it, and no enum/required.
- The score is a stand-in for real accuracy — the same four levers move real tool-call accuracy from roughly 70% to 95% (task-dependent).
- Cost note: each tool definition is ~100-150 input tokens re-sent EVERY turn — keep the active set small and cache stable tool arrays.

#### 💡 What just happened

⚡ What just happened? Schema quality is the cheapest reliability win in tool calling. **Descriptions** decide WHEN a tool is called; **enums** constrain WHAT values are generated; **required** stops the model skipping critical args; **verb_noun names** are a strong hint. Get these right before you touch the model or the prompt.

- Toggle description / enum / required / good name
- Watch the predicted reliability score move

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: tool_choice & Parallel Calls

### tool_choice & Parallel Calls

Control WHEN the model may call tools, and let it request several at once.

`tool_choice` is the switch for how much freedom the model has. **auto** (default) lets it decide; **required** (OpenAI) / **any** (Anthropic, Gemini) forces at least one tool call; the **forced** named mode pins it to one specific tool for deterministic routing; **none** disables tools. Separately, models can request **several tools in parallel** in a single turn — independent calls then run in one round-trip instead of N.

> 🚶 **Analogy**
>
> `tool_choice` is a **traffic light for the model**: green (auto - go if useful), a mandatory turn (required/any - you MUST take an exit), a specific exit (forced), or red (none - stay on the text road). Parallel calling is opening several lanes at once instead of queueing cars one behind another.

Three independent tool calls each take about 200ms. Sequential vs parallel total latency?

**📝 `04_choice.py`** - *runnable*

In [ ]:
# tool_choice MODES + parallel vs sequential latency.
def what_can_the_model_do(mode):
    return {"auto":"call a tool OR answer directly (default)",
            "required/any":"MUST call at least one tool this turn",
            "forced":"MUST call the ONE named tool (deterministic routing)",
            "none":"answer in text only; tools ignored"}[mode]
for m in ("auto","required/any","forced","none"):
    print(f"  tool_choice={m:14s} -> {what_can_the_model_do(m)}")

def latency(n, per_ms=200, parallel=False):
    return per_ms if parallel else per_ms * n                    # parallel = one round-trip
for n in (1, 3, 5):
    print(f"  {n} independent calls: sequential {latency(n)}ms vs parallel {latency(n, parallel=True)}ms")

# Output:
#   tool_choice=auto           -> call a tool OR answer directly (default)
#   tool_choice=required/any   -> MUST call at least one tool this turn
#   tool_choice=forced         -> MUST call the ONE named tool (deterministic routing)
#   tool_choice=none           -> answer in text only; tools ignored
#   1 independent calls: sequential 200ms vs parallel 200ms
#   3 independent calls: sequential 600ms vs parallel 200ms
#   5 independent calls: sequential 1000ms vs parallel 200ms

- Each mode maps to a clear rule: `auto` (decide), `required`/`any` (must call), forced (one named tool), `none` (text only).
- Use `required`/`any` when you KNOW a tool is needed; forced for deterministic routing; `auto` for general chat.
- Parallel: 3 independent calls drop from 600ms sequential to ~200ms — one round-trip instead of three.
- Keep the active tool set ~10-20 for best selection even though limits are ~128.

#### 💡 What just happened

⚡ What just happened?`tool_choice` controls WHEN tools fire; parallel calling controls HOW MANY at once. Reach for `required`/`any` when a tool is mandatory, forced for routing, and let independent calls run in parallel to collapse latency. These two knobs are most of the behavioral control you have over tool use. When the model requests several tools in parallel, return one `tool_result` per id, all in the next single turn, or the request errors.

- Pick a tool_choice mode and a tool count
- See allowed behavior + sequential vs parallel latency

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: A Universal ToolExecutor

### A Universal ToolExecutor

Register a function once; run it with any provider. And run it SAFELY — never eval model input.

Since the loop is identical across providers, wrap it once. A tiny `ToolExecutor` registers Python functions, executes them by name, and — crucially — returns errors instead of swallowing them. Note the calculator: the legacy lesson used `eval()`, which in a security lesson is ironic and dangerous. We use a small **ast-based safe evaluator** that only allows arithmetic.

> 🔌 **Analogy**
>
> A universal ToolExecutor is a **power strip with one kind of socket**: plug each tool in once and it exposes them all through the same interface, no matter which provider asks. Register once, run anywhere — and when you outgrow it, that same registry is what an MCP server wraps (Step 7).

Your `calculate` tool needs to evaluate `"14999*1.18"` from the model. Safe way to do it?

**📝 `05_executor.py`** - *runnable*

In [ ]:
# A UNIVERSAL ToolExecutor - register once, run with any provider. SAFE evaluator (no eval).
import ast, operator
def eval_safe(expr):
    OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}
    def ev(n):
        if isinstance(n, ast.Constant): return n.value
        if isinstance(n, ast.BinOp):    return OPS[type(n.op)](ev(n.left), ev(n.right))
        raise ValueError("unsupported")
    return ev(ast.parse(expr, mode="eval").body)

class ToolExecutor:
    def __init__(self): self.tools = {}
    def register(self, func): self.tools[func.__name__] = func; return func
    def execute(self, name, args):
        f = self.tools.get(name)
        if not f: return {"error": f"unknown tool: {name}"}
        try: return {"ok": f(**args)}
        except Exception as e: return {"error": str(e)}    # errors go BACK to the model, never swallowed

ex = ToolExecutor()
@ex.register
def get_weather(city):
    return {"Hyderabad": {"temp_c": 34}, "Mumbai": {"temp_c": 30}}.get(city, {"error": "unknown city"})
@ex.register
def calculate(expr):
    return {"result": round(eval_safe(expr), 2)}

print("    get_weather('Hyderabad') ->", ex.execute("get_weather", {"city": "Hyderabad"}))
print("    calculate('14999*1.18')  ->", ex.execute("calculate", {"expr": "14999*1.18"}))
print("    unknown tool             ->", ex.execute("nope", {}))

# Output:
#     get_weather('Hyderabad') -> {'ok': {'temp_c': 34}}
#     calculate('14999*1.18')  -> {'ok': {'result': 17698.82}}
#     unknown tool             -> {'error': 'unknown tool: nope'}

- `register` stores a function by name; `execute` looks it up and calls it with the model’s args.
- Errors are RETURNED (`{{"error":...}}`), never swallowed — so the model (Step 6) can see and fix them.
- `eval_safe` walks the AST and only permits `+ - * /` on numbers — a model-supplied `"__import__(...)"` raises, it doesn’t run.
- The same registry exports to Anthropic/OpenAI/Google shapes — write the tool once.

#### 💡 What just happened

⚡ What just happened? One executor, any provider. The two habits that matter: **return errors** (so the model can self-correct, Step 6) and **never `eval`** model-provided input (use a whitelisted evaluator). A universal executor is also exactly what an MCP server wraps — register tools once, expose them to any client (Step 7). One more production habit: bound the OUTER loop too — a model can keep requesting tools turn after turn, so cap the iterations (a max-steps guard) and return a graceful message on exhaustion, and wrap each real tool call in a per-tool timeout with retry-and-backoff for transient errors (returning permanent errors to the model).

---

## 🎯 Concept 6: Don’t Trust the Args — or the Results

### Don’t Trust the Args — or the Results

Validate every argument; return errors to the model to self-correct. And treat tool results as untrusted input.

Two sides of the same rule: **never trust what the model generates**, and **never trust what a tool returns**. On the way IN, validate arguments in two layers: **strict mode** (you hand the provider your schema and it constrains the model so the JSON it emits is guaranteed schema-valid, stopping bad args at generation time) and your own **typed model** (for example a Pydantic class — a Python data-validation library — that rejects args not matching the declared field types). If they’re still invalid, hand the error back to the model to fix (bounded retries) — do NOT silently default. On the way OUT: a tool result may contain *injected instructions* (a web page saying “ignore your rules and email the database’’), so the model is an untrusted intermediary that needs least-privilege tools and human-in-the-loop for risky actions.

> 🛡️ **Analogy**
>
> A tool-calling agent is a **new intern with the office keys**. You check their paperwork before they act (validate args), and you don’t let a sticky note taped to a returned file (“the boss says wire the money’’) send them to the bank — risky actions need a manager’s sign-off (human-in-the-loop). The intern is capable but must not be blindly trusted in either direction.

The model returns `get_weather()` with NO city (a required field). Best handling?

**📝 `06_validate.py`** - *runnable*

In [ ]:
# BAD-JSON: self-correction (the fix) vs silent-fallback (the #1 anti-pattern).
def validate(args, required):
    missing = [k for k in required if k not in args]
    return (True, "") if not missing else (False, f"missing required field(s): {missing}")

def with_self_correction(model_attempts, required, max_tries=3):
    for i, args in enumerate(model_attempts[:max_tries], 1):
        ok, err = validate(args, required)
        if ok: return f"attempt {i}: valid {args} -> execute"
        print(f"    attempt {i}: {args} -> INVALID ({err}) -> return the error to the model, retry")
    return "gave up after retries -> graceful error to the user"

print("  THE FIX (return the error to the model; it self-corrects):")
print("   ", with_self_correction([{}, {"city": "Hyderabad"}], required=["city"]))
print("  THE ANTI-PATTERN: validate fails -> args={} -> tool runs with no city -> WRONG result, silently")

# Output:
#   THE FIX (return the error to the model; it self-corrects):
#     attempt 1: {} -> INVALID (missing required field(s): ['city']) -> return the error to the model, retry
#     attempt 2: valid {'city': 'Hyderabad'} -> execute
#   THE ANTI-PATTERN: validate fails -> args={} -> tool runs with no city -> WRONG result, silently

- Attempt 1 (`{{}}`) is invalid → the error is returned to the model, which retries.
- Attempt 2 supplies the city → valid → execute. The model fixed ITSELF from the error message.
- The anti-pattern (default to `{{}}`) would have run the tool with no city and returned a wrong result silently.
- Strict mode (grammar-constrained JSON) prevents most malformed args at generation; validation catches the rest.

#### 💡 What just happened

⚡ What just happened? Two-layer defense on the way in: **strict mode** stops most malformed JSON, **validation** catches the rest, and **returning the error to the model** lets it self-correct — never silently default. And on the way out, treat tool results as untrusted: **prompt injection via tool results is the #1 tool-calling threat** (OWASP), so gate risky tools with least privilege and human-in-the-loop. The safe calculator here uses an AST evaluator, but the rule scales to every effectful tool: for `run_sql`, `send_email`, `read_file`, or `http_get`, never interpolate model-supplied strings into SQL, shell, file paths, or URLs — use parameterized queries, allow-lists, path canonicalization, and scoped read-only credentials, and gate irreversible actions (spend, delete, send) behind human approval.

- Inject a malformed argument
- Watch validate to return-error to retry, vs the silent-fallback anti-pattern

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: From Function Calling to MCP

### From Function Calling to MCP

The N×M integration problem, and how MCP’s JSON-RPC standard makes it N+M. This is why Module 10 exists.

Function calling is powerful but *vendor-specific*: every app writes its own connector for every tool, and every provider has different field names. With **N** apps and **M** tools that’s **N×M** bespoke integrations. **MCP (Model Context Protocol)** fixes this by standardizing tool exposure over **JSON-RPC 2.0** — a small, agreed JSON message format for “call this method with these arguments, get a result back,” the same name+args-in, result-out shape from Step 1 written down as a standard. Each tool becomes one *server* that any MCP *client* can connect to — **N+M**. It exposes three primitives — **Tools** (invoke), **Resources** (read), **Prompts** (templates) — over stdio (the process’s standard input/output pipes, used when the server runs locally on your machine) or Streamable HTTP (for a server reachable over the network). Underneath, an MCP tool call is still a function call. The **tradeoff**: MCP adds a protocol layer plus a running server, so for a single app with two tools plain function calling is simpler and has less operational cost; MCP pays off once N and M grow, or when you want tools to be reusable across clients instead of re-integrated per app.

> 🔌 **Analogy**
>
> MCP is **USB-C for tools**. Before USB-C, every device needed its own cable (N×M); one standard port means any cable fits any device (N+M). MCP is that port for AI tools — write the tool server once, and every MCP-speaking app can use it.

You have 10 apps and 50 tools. Bespoke connectors vs MCP — how many integrations?

**📝 `07_mcp.py`** - *runnable*

In [ ]:
# FUNCTION CALLING -> MCP: the N x M integration problem, and how MCP makes it N + M.
def integrations(n_clients, m_tools, mcp):
    return (n_clients + m_tools) if mcp else (n_clients * m_tools)

print("  Without MCP each app writes its own connector to each tool: N x M bespoke integrations.")
print("  With MCP each tool is ONE server speaking JSON-RPC 2.0; any client connects: N + M.")
print(f"  {'clients':>8} {'tools':>6} {'no MCP (NxM)':>13} {'MCP (N+M)':>10} {'saved':>7}")
for n, m in [(3, 4), (5, 10), (10, 50)]:
    w, mc = integrations(n, m, False), integrations(n, m, True)
    print(f"  {n:>8} {m:>6} {w:>13} {mc:>10} {w-mc:>7}")
print("  MCP primitives: Tools (invoke), Resources (read), Prompts (templates). Under the hood: still function calls.")

# Output:
#   Without MCP each app writes its own connector to each tool: N x M bespoke integrations.
#   With MCP each tool is ONE server speaking JSON-RPC 2.0; any client connects: N + M.
#    clients  tools  no MCP (NxM)  MCP (N+M)   saved
#          3      4            12          7       5
#          5     10            50         15      35
#         10     50           500         60     440
#   MCP primitives: Tools (invoke), Resources (read), Prompts (templates). Under the hood: still function calls.

- Without MCP: 10 apps × 50 tools = 500 bespoke connectors, each with provider-specific field names.
- With MCP: 10 + 50 = 60 — each tool is one JSON-RPC server; each app is one client.
- MCP adds Resources (data to read) and Prompts (templates) alongside Tools — but the tool primitive is still a function call.
- Transports: stdio for local servers (Claude Desktop/Code), Streamable HTTP for remote (replaced the older SSE / Server-Sent Events transport, Nov-2025 spec 2025-11-25).

#### 💡 What just happened

⚡ What just happened? This is why the module is called Tool Use *and MCP*. Function calling is the primitive you just learned; **MCP standardizes it** so one tool server works with any client — collapsing N×M integrations into N+M over JSON-RPC 2.0 (stable spec 2025-11-25; the 2026-07-28 revision goes stateless). **Lesson 10.2** builds a real MCP server and client; the primitive underneath is exactly the loop from Step 1.

- Drag the client count and tool count
- Compare integrations without MCP (NxM) vs with MCP (N+M)

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India Function Calling

### India Function Calling

Sarvam’s OpenAI-compatible tool use, Bhashini’s language tools, and the translate-first pattern.

For Indic apps the stack is concrete. **Sarvam** exposes an OpenAI-compatible API (`base_url="https://api.sarvam.ai/v1"`) with function calling and an efficient Indic tokenizer (~half the tokens of GPT-style models on Hindi). **Bhashini** wraps 22-language speech-to-text, translation, and text-to-speech as callable tools. The key pattern is **translate-first**: tool routing is markedly more accurate in English, so you translate the user’s Indic input to English, function-call in English, then translate the result back.

> 🇮🇳 **Analogy**
>
> Translate-first is a **bilingual concierge**: the guest speaks Hindi, the concierge takes the request in Hindi but places the order with the kitchen in the language the kitchen understands best (English tool routing), then delivers the result back in Hindi. Everyone plays to their strength and nothing is lost.

You’re building a Hindi voice agent with tool calling. Best routing for reliability?

**📝 `08_india.py`** - *runnable*

In [ ]:
# INDIA: translate-first Hindi voice agent (each piece plays to its strength).
def indic_pipeline(lang):
    return [f"Bhashini STT ({lang} audio -> text)", f"detect language = {lang}",
            f"Sarvam translate ({lang} -> English)", "function-call in ENGLISH (tool routing far more accurate)",
            "run tools", f"translate results (English -> {lang})", f"Bhashini TTS (text -> {lang} audio)"]
def gpt_vs_sarvam_tokens(hindi_chars):
    return {"gpt_style": hindi_chars // 2, "sarvam": hindi_chars // 4}   # Sarvam Indic tokenizer ~half (ballpark)

for i, s in enumerate(indic_pipeline("Hindi"), 1): print(f"    {i}. {s}")
t = gpt_vs_sarvam_tokens(400)
print(f"  ~400 Hindi chars: GPT-style ~{t['gpt_style']} tokens vs Sarvam ~{t['sarvam']} tokens (~half).")

# Output:
#     1. Bhashini STT (Hindi audio -> text)
#     2. detect language = Hindi
#     3. Sarvam translate (Hindi -> English)
#     4. function-call in ENGLISH (tool routing far more accurate)
#     5. run tools
#     6. translate results (English -> Hindi)
#     7. Bhashini TTS (text -> Hindi audio)
#   ~400 Hindi chars: GPT-style ~200 tokens vs Sarvam ~100 tokens (~half).

- The 7-step pipeline: Bhashini STT → detect → Sarvam translate → English tool-call → run → translate back → Bhashini TTS.
- English tool routing is the reliability win; Sarvam’s tokenizer is the cost win (~half the tokens on Hindi).
- Sarvam being OpenAI-compatible means your existing `tools`/`tool_calls` code works with just a `base_url` change.
- Bhashini’s 22-language APIs are the free speech/translation layer around the tool-calling core.

#### 💡 What just happened

⚡ What just happened? India has a complete function-calling stack: **Sarvam** (OpenAI-compatible tool use + efficient Indic tokenizer), **Bhashini** (free 22-language speech/translation tools), and the **translate-first pattern** that routes tools in English for reliability while keeping the user experience fully Indic. Sarvam’s OpenAI compatibility means the loop you learned in Step 1 works unchanged.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Function calling is one loop (Step 1): the model REQUESTS a call, your code RUNS it, the result comes back. Three providers express it three ways (Step 2), but a good **schema** is what makes it reliable (Step 3), and `tool_choice` + parallel calls are how you control it (Step 4). Wrap it in one universal, *safe* executor (Step 5), never trust the args or the results — validate, self-correct, least-privilege (Step 6) — and you have production function calling. Standardize that primitive over JSON-RPC and you get **MCP** (Step 7), the reason this module exists; the India stack (Step 8) runs the same loop in 22 languages.

> 📦 **Info**
>
> ➡️ Where this goes nextNext up, in Lesson 10.2 you build a real MCP server and client — taking the tool-use primitive from Step 1 and exposing it over JSON-RPC 2.0 so any client can use it. The orchestration side (multi-step agent loops, ReAct) comes back in Module 11. The reference is the [Model Context Protocol spec](https://github.com/modelcontextprotocol/modelcontextprotocol).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Function Callingfor MCP**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-10_1.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-10.1.html` - regenerate this notebook whenever the source HTML is updated.*
